# Model implementation on tunder

In this notebook, we presented how we adapt the continous integration (MS2) to work on tunder machine.

<a id="table"></a>

## Table of Contents
- [Model explanation](#model-explanation)

    - [Computing capacity required](#power)
    - [Setup the envtioment](#envrioment)
    - [Bash script to process all the analysis](#bash)
    - [Satellite images download](#satellite)
    - [Transposing the dataset and adding date](#transpose)
    - [Merging historical with newly acquired data](#merge)
    - [Analysis](analysis#)
    - [Tiff creation](#tiff)



<a id="power"></a>

[Go back to the table](#table)

## Computing capacity required

TODO: See the most demanding dask worker

<a id="envrioment"></a>

[Go back to the table](#table)

## Setup the envtioment

The envrioment is already avaible in the folder, it is possible to activate with the command

TODO: add command

It possible to setup the envrioment using Conda or Pip. the following command will create the envrioment based on the preferred choice.

TODO: add command

Once the envrioment is created, it can be activated following the command above

<a id="bash"></a>

[Go back to the table](#table)

## Bash script to process all the analysis

We created a bash script that automatically launch all the script in sequence. The script will perform the following action:

- retrieve the last date analysed by the model
- set the current date as the last date to analyse
- exectue the script 1 to search and download the new satellite images
- if no satellite images are detected, it skip all the computation and overwrite the last date to the current date
- if one (or more) satellite images are found, exectue in order all the script from 2 to 6
- after the computation are finished, it overwrites the last date to the current date

We introduce this dynamic windows of last date analysed and current date to have a system that is flexible (can be aither run once per day or with some days lag).


<a id="satellite"></a>

[Go back to the table](#table)

## Satellite images download

The satellite images donwload follows the same logic as the prior version.
It 

TODO: explain new coordinates


<a id="transpose"></a>

[Go back to the table](#table)

## Transposing the dataset and adding date

TODO


<a id="merge"></a>

[Go back to the table](#table)

## Merging historical with newly acquired data

The merging of previously analysed data (historical) and the new acquired data.

The zarr dataset will have the following information:

- Data

    - NDVI timeserie (pixel X time)
    - Median NDVI obtained from Samantha model (pixel X time)
    - Mask specifying the information of NDVI value (pixel X time)
    - Boolean array specifying if a date has some observation or not (time)

- Coordinates

    - pixel
    - date
    - X and Y position on the dataset
    - X and Y coordinates used to create the TIFF for each time layer

The folder have a size of TODO


<a id="analysis"></a>

[Go back to the table](#table)

## Analysis

The analysis performed on tunder uses the function described in TODO

Here there are the summarised step of the function, which is applied for each pixel indipendentely.

To smooth a value is needed a 7 windows of observation and the value of interest must be at the center, we load the timeseries from the last third observation up to the current date.

- Step 1: load the data from the last third observation up to the current date

- Step 2: filter the data, including only the observation data (the data must be an observation and have a value between 0 and 1)

- Step 3: perform the outlier detection based on 

    - the difference between the NDVI and corresponded median value

    - the difference between the current delta (defined as the difference between the NDVI and corresponded median value and the neighoubring deltas)

- If both conditions are met, the data is an outlier and is removed from the analysis

- To perform the ...

- For each date in the remaining timeserie, we evaluate a rolling window of 7 values to find an extreme negative NDVI anomalies or values close to the boundaries condition:

    - if at least 5 out 7 values have a delta of -0.2 or the NDVI values is above 0.95 or below 0.05, we skip the smoothing and the linear interpolation is performed
    - if the case above does not occour, the smoothing of the fourth value (in the middle is performed)

- After the smoothing of the rolling window is applied, the deltas are linearly interpolated to the full time series and summed to the median NDVI to obtained the processed NDVI values

- The mask NDVI array is created based on ...

- The data are wrote back to be used for the TIFF generation and re-used for the merging of the future data

TODO: benchmark

<a id="tiff"></a>

[Go back to the table](#table)

## Tiff creation

TODO